# LeetCode 2315 — Count Asterisks bằng Python

Link đề: https://leetcode.com/problems/count-asterisks/ · Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def countAsterisks(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: các dấu gạch đứng `|` chia chuỗi thành **các cặp** (1 với 2, 3 với 4, ...), và ta chỉ đếm dấu `*` **ngoài** các cặp đó.
2. Biết kỹ thuật **cờ bật/tắt (toggle flag)** hoặc **đếm chẵn/lẻ (parity)** để biết mình đang "trong" hay "ngoài" cặp.
3. Biết dùng `str.split('|')`, `str.count('*')`, lát cắt bước nhảy `[::2]` và biểu thức chính quy `re.sub`.
4. Viết 5 cách: duyệt từng ký tự với cờ, `split` rồi lấy các đoạn chỉ số chẵn, đếm `|` đã gặp (parity), regex xóa các cặp, và cắt chuỗi theo `find`.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: đếm cả `*` bên trong cặp, cập nhật cờ ở sai thời điểm, hiểu sai "cặp" khi chuỗi có nhiều gạch.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ thường, dấu `*` và dấu `|`. Các dấu `|` được ghép cặp **theo thứ tự**: `|` thứ 1 với thứ 2, thứ 3 với thứ 4, thứ 5 với thứ 6, ... Trả về số dấu `*` **không nằm giữa** một cặp `|`.

Chú ý: số lượng dấu `|` trong `s` luôn là số **chẵn**.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"l\|*e*et\|c**o\|*de\|"` | `2` | các đoạn được tính là ngoài cặp: `"l"`, `"c**o"` (2 dấu `*`), và đoạn rỗng cuối. Dấu `*` trong `"*e*et"` và `"*de"` bị bỏ |
| 2 | `"iamprogrammer"` | `0` | không có dấu `*` |
| 3 | `"yo\|uar\|e**\|b\|e***au\|tifu\|l"` | `5` | ngoài cặp: `"yo"`, `"e**"`, `"e***au"`, `"l"` -> `2 + 3 = 5` |

Constraints: `1 <= len(s) <= 1000`; `s` chỉ gồm chữ thường, `'|'` và `'*'`; số dấu `|` là số chẵn.

In [1]:
examples = [("l|*e*et|c**o|*de|", 2), ("iamprogrammer", 0), ("yo|uar|e**|b|e***au|tifu|l", 5)]
for s, expected in examples:
    parts = s.split("|")
    labels = ["ngoài" if i % 2 == 0 else "trong" for i in range(len(parts))]
    print(f"s={s!r}")
    for p, lab in zip(parts, labels):
        print(f"    {lab}: {p!r} có {p.count('*')} dấu *")
    print("    mong đợi", expected)

s='l|*e*et|c**o|*de|'
    ngoài: 'l' có 0 dấu *
    trong: '*e*et' có 2 dấu *
    ngoài: 'c**o' có 2 dấu *
    trong: '*de' có 1 dấu *
    ngoài: '' có 0 dấu *
    mong đợi 2
s='iamprogrammer'
    ngoài: 'iamprogrammer' có 0 dấu *
    mong đợi 0
s='yo|uar|e**|b|e***au|tifu|l'
    ngoài: 'yo' có 0 dấu *
    trong: 'uar' có 0 dấu *
    ngoài: 'e**' có 2 dấu *
    trong: 'b' có 0 dấu *
    ngoài: 'e***au' có 3 dấu *
    trong: 'tifu' có 0 dấu *
    ngoài: 'l' có 0 dấu *
    mong đợi 5


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi từng ký tự và so sánh ký tự

Chuỗi duyệt được bằng `for ch in s`. Mỗi `ch` là một chuỗi dài 1, so sánh bằng `==`. Ký tự `'|'` và `'*'` là hai ký tự đặc biệt bình thường trong Python (không cần escape trong chuỗi thường).

In [2]:
s = "a|*b|*"
for ch in s:
    print(repr(ch), ch == "|", ch == "*")

'a' False False
'|' True False
'*' False True
'b' False False
'|' True False
'*' False True


### 3.2. Cờ bật/tắt (toggle flag)

Dùng biến bool `inside`. Mỗi lần gặp `|` thì **đảo** `inside = not inside`. Khi `inside == False` ta đang ở ngoài cặp. Đây là cách mô hình hóa "mở/đóng" ngoặc: lần `|` lẻ là mở, lần chẵn là đóng.

In [3]:
inside = False
for ch in "a|b|c":
    if ch == "|":
        inside = not inside
    print(repr(ch), "inside =", inside)

'a' inside = False
'|' inside = True
'b' inside = True
'|' inside = False
'c' inside = False


### 3.3. `str.split` và `str.count`

`s.split('|')` cắt chuỗi tại mỗi `|` và trả list các đoạn. Nếu có `2k` dấu `|` thì có `2k + 1` đoạn; các đoạn ở chỉ số **chẵn** (`0, 2, 4, ...`) nằm **ngoài** cặp, chỉ số **lẻ** nằm **trong** cặp. `str.count('*')` đếm số dấu `*` trong một đoạn.

In [4]:
s = "l|*e*et|c**o|*de|"
parts = s.split("|")
print(parts)
print(len(parts))
print(parts[0::2])                 # các đoạn ngoài cặp
print([p.count("*") for p in parts[0::2]])

['l', '*e*et', 'c**o', '*de', '']
5
['l', 'c**o', '']
[0, 2, 0]


### 3.4. Lát cắt bước nhảy `[::2]` và `sum`

`parts[::2]` lấy các phần tử chỉ số `0, 2, 4, ...`. Kết hợp `sum(p.count('*') for p in parts[::2])` cho đáp án trong một dòng.

In [5]:
parts = ["a", "b", "c", "d", "e"]
print(parts[::2], parts[1::2])
print(sum(len(p) for p in parts[::2]))

['a', 'c', 'e'] ['b', 'd']
3


### 3.5. Biểu thức chính quy cơ bản với `re.sub`

`re.sub(pattern, repl, s)` thay mọi chỗ khớp. Mẫu `\|[^|]*\|` nghĩa là: một `|`, rồi **bất kỳ ký tự nào không phải `|`** (zero hay nhiều), rồi một `|`. Mẫu này khớp đúng một **cặp** và nội dung bên trong nó. Xóa các cặp xong, phần còn lại chính là phần **ngoài** cặp.

In [6]:
import re

s = "l|*e*et|c**o|*de|"
print(re.sub(r"\|[^|]*\|", "", s))
print(re.sub(r"\|[^|]*\|", "", s).count("*"))

lc**o
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hình dung các dấu `|` như **cổng**. Cổng 1 mở một "phòng", cổng 2 đóng phòng. Cổng 3 mở phòng mới, cổng 4 đóng... Mọi dấu `*` **trong phòng** bị bỏ qua; ta chỉ đếm `*` ở **hành lang** (ngoài phòng).

Tính tay `"yo|uar|e**|b|e***au|tifu|l"`:

- `yo` — hành lang, không có `*`.
- cổng 1 `|` — vào phòng; `uar` trong phòng.
- cổng 2 `|` — ra hành lang; `e**` có **2** dấu `*`.
- cổng 3 `|` — vào phòng; `b` trong phòng.
- cổng 4 `|` — ra; `e***au` có **3** dấu `*`.
- cổng 5 `|` — vào; `tifu` trong phòng.
- cổng 6 `|` — ra; `l` không có `*`.
- Tổng `2 + 3 = 5`.

In [7]:
s = "yo|uar|e**|b|e***au|tifu|l"
inside, total = False, 0
for ch in s:
    if ch == "|":
        inside = not inside
        print(f"gặp '|' -> {'vào phòng' if inside else 'ra hành lang'}")
    elif ch == "*" and not inside:
        total += 1
        print(f"  * ở hành lang -> total={total}")
print("tổng:", total)

gặp '|' -> vào phòng
gặp '|' -> ra hành lang
  * ở hành lang -> total=1
  * ở hành lang -> total=2
gặp '|' -> vào phòng
gặp '|' -> ra hành lang
  * ở hành lang -> total=3
  * ở hành lang -> total=4
  * ở hành lang -> total=5
gặp '|' -> vào phòng
gặp '|' -> ra hành lang
tổng: 5


## 5. Cách 1 — Brute Force / baseline: duyệt từng ký tự với cờ

**Ý tưởng:** biến `inside = False`. Duyệt `s`; gặp `|` thì đảo `inside`; gặp `*` mà `inside == False` thì tăng đếm.

**Độ phức tạp:** một lượt `O(n)`, bộ nhớ `O(1)`. Đây đã là tối ưu về bậc; các cách sau chỉ khác cách viết, không nhanh hơn về bậc. (Nút thắt không có; ta học thêm nhiều cách nhìn để nhận ra pattern.)

In [8]:
class SolutionFlag:
    def countAsterisks(self, s: str) -> int:
        inside = False
        count = 0
        for ch in s:
            if ch == "|":
                inside = not inside
            elif ch == "*" and not inside:
                count += 1
        return count

for s in ("l|*e*et|c**o|*de|", "iamprogrammer", "yo|uar|e**|b|e***au|tifu|l"):
    print(repr(s), "->", SolutionFlag().countAsterisks(s))

'l|*e*et|c**o|*de|' -> 2
'iamprogrammer' -> 0
'yo|uar|e**|b|e***au|tifu|l' -> 5


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "trong cặp hay ngoài cặp" chỉ phụ thuộc vào **số dấu `|` đã gặp là chẵn hay lẻ**:

- đã gặp số **chẵn** dấu `|` (0, 2, 4, ...) -> **ngoài** cặp;
- đã gặp số **lẻ** dấu `|` -> **trong** cặp.

Do đó ta không cần cờ: cứ đếm `bars` rồi xét `bars % 2 == 0`. Cũng suy ra: `s.split('|')` cho các đoạn mà đoạn chỉ số chẵn là ngoài cặp.

**Invariant:** trước khi xử lý ký tự `s[i]`, `bars` = số `|` trong `s[:i]`, và `inside == (bars % 2 == 1)`.

**Quy tắc:** `|` -> `bars += 1`; `*` và `bars` chẵn -> `count += 1`.

## 7. Các cách tiếp theo

### Cách 2 — `split('|')` rồi chỉ đếm các đoạn chỉ số chẵn

Gọi `parts = s.split('|')`. Đoạn `parts[0]`, `parts[2]`, ... nằm ngoài cặp. Đếm `*` trong từng đoạn đó và cộng lại.

In [9]:
class SolutionSplit:
    def countAsterisks(self, s: str) -> int:
        return sum(part.count("*") for part in s.split("|")[::2])

for s in ("l|*e*et|c**o|*de|", "iamprogrammer", "yo|uar|e**|b|e***au|tifu|l"):
    print(repr(s), "->", SolutionSplit().countAsterisks(s))

'l|*e*et|c**o|*de|' -> 2
'iamprogrammer' -> 0
'yo|uar|e**|b|e***au|tifu|l' -> 5


**Độ phức tạp Cách 2:** Time `O(n)`, Space `O(n)` cho danh sách các đoạn.

### Cách 3 — Đếm `|` đã gặp (parity), không cần cờ bool

Giữ `bars` là số `|` đã gặp. Kiểm tra `bars % 2 == 0` (hoặc `bars & 1 == 0`).

In [10]:
class SolutionParity:
    def countAsterisks(self, s: str) -> int:
        bars = count = 0
        for ch in s:
            if ch == "|":
                bars += 1
            elif ch == "*" and bars % 2 == 0:
                count += 1
        return count

for s in ("l|*e*et|c**o|*de|", "a*|*|*|*|*", "||"):
    print(repr(s), "->", SolutionParity().countAsterisks(s))

'l|*e*et|c**o|*de|' -> 2
'a*|*|*|*|*' -> 3
'||' -> 0


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Biểu thức chính quy: xóa các cặp rồi đếm

`re.sub(r"\|[^|]*\|", "", s)` xóa mọi `|...|` (trong đó `...` không chứa `|`). Vì các `|` ghép cặp liên tiếp nên regex này xóa **đúng các cặp**, rồi `.count('*')` đếm phần còn lại.

In [11]:
import re

class SolutionRegex:
    def countAsterisks(self, s: str) -> int:
        return re.sub(r"\|[^|]*\|", "", s).count("*")

for s in ("l|*e*et|c**o|*de|", "a*|*|*|*|*", "||"):
    print(repr(s), "->", SolutionRegex().countAsterisks(s))

'l|*e*et|c**o|*de|' -> 2
'a*|*|*|*|*' -> 3
'||' -> 0


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)` cho chuỗi mới. Regex ngắn nhưng khó đọc với người mới; hiểu mẫu `\|[^|]*\|` là chìa khóa.

### Cách 5 — Dùng `find` để nhảy từ cặp này sang cặp kia

Dùng `str.find('|', start)` tìm vị trí `|` kế tiếp. Ta lặp: đếm `*` trong đoạn từ vị trí hiện tại tới `|` mở; nhảy tới `|` đóng và tiếp tục. Cách này không duyệt ký tự trong cặp.

In [12]:
class SolutionFind:
    def countAsterisks(self, s: str) -> int:
        count, pos = 0, 0
        while True:
            open_i = s.find("|", pos)
            if open_i == -1:                       # không còn cặp nữa: đếm đoạn đuôi
                count += s.count("*", pos)
                return count
            count += s.count("*", pos, open_i)     # đoạn ngoài cặp
            close_i = s.find("|", open_i + 1)      # dấu | đóng (luôn tồn tại vì số | chẵn)
            pos = close_i + 1

for s in ("l|*e*et|c**o|*de|", "iamprogrammer", "yo|uar|e**|b|e***au|tifu|l"):
    print(repr(s), "->", SolutionFind().countAsterisks(s))

'l|*e*et|c**o|*de|' -> 2
'iamprogrammer' -> 0
'yo|uar|e**|b|e***au|tifu|l' -> 5


**Độ phức tạp Cách 5:** Time `O(n)` (mỗi ký tự được `find`/`count` quét một lần), Space `O(1)`. `s.count(sub, start, end)` chỉ đếm trong `s[start:end]` mà không sao chép chuỗi.

## 8. TRACE từng bước

Trace Cách 3 (`bars` chẵn/lẻ) với `s = "l|*e*et|c**o|*de|"`:

| Ký tự | `bars` trước | hành động | `bars` sau | `count` |
|---|---|---|---|---|
| `l` | 0 | không | 0 | 0 |
| `\|` | 0 | `bars += 1` | 1 | 0 |
| `*` | 1 | lẻ -> trong cặp, bỏ | 1 | 0 |
| `e` | 1 | không | 1 | 0 |
| `*` | 1 | trong cặp, bỏ | 1 | 0 |
| `e`,`t` | 1 | không | 1 | 0 |
| `\|` | 1 | `bars += 1` | 2 | 0 |
| `c` | 2 | không | 2 | 0 |
| `*` | 2 | chẵn -> ngoài, đếm | 2 | 1 |
| `*` | 2 | đếm | 2 | 2 |
| `o` | 2 | không | 2 | 2 |
| `\|` | 2 | `bars += 1` | 3 | 2 |
| `*` | 3 | trong cặp, bỏ | 3 | 2 |
| `d`,`e` | 3 | không | 3 | 2 |
| `\|` | 3 | `bars += 1` | 4 | 2 |

Kết quả `2`.

In [13]:
def trace_parity(s):
    bars = count = 0
    print(f"s={s!r}")
    for i, ch in enumerate(s):
        if ch == "|":
            bars += 1
            print(f"  i={i:2}: '|'  bars={bars} -> {'trong' if bars % 2 else 'ngoài'} cặp")
        elif ch == "*":
            ngoai = bars % 2 == 0
            if ngoai:
                count += 1
            print(f"  i={i:2}: '*'  bars={bars} ({'ngoài' if ngoai else 'trong'}) count={count}")
    return count

print("kết quả:", trace_parity("l|*e*et|c**o|*de|"))

s='l|*e*et|c**o|*de|'
  i= 1: '|'  bars=1 -> trong cặp
  i= 2: '*'  bars=1 (trong) count=0
  i= 4: '*'  bars=1 (trong) count=0
  i= 7: '|'  bars=2 -> ngoài cặp
  i= 9: '*'  bars=2 (ngoài) count=1
  i=10: '*'  bars=2 (ngoài) count=2
  i=12: '|'  bars=3 -> trong cặp
  i=13: '*'  bars=3 (trong) count=2
  i=16: '|'  bars=4 -> ngoài cặp
kết quả: 2


In [14]:
def trace_split(s):
    parts = s.split("|")
    total = 0
    for i, p in enumerate(parts):
        side = "ngoài" if i % 2 == 0 else "trong"
        c = p.count("*")
        if i % 2 == 0:
            total += c
        print(f"  parts[{i}]={p!r:10} {side:5} có {c} dấu * -> total={total}")
    return total

print("kết quả:", trace_split("yo|uar|e**|b|e***au|tifu|l"))

  parts[0]='yo'       ngoài có 0 dấu * -> total=0
  parts[1]='uar'      trong có 0 dấu * -> total=0
  parts[2]='e**'      ngoài có 2 dấu * -> total=2
  parts[3]='b'        trong có 0 dấu * -> total=2
  parts[4]='e***au'   ngoài có 3 dấu * -> total=5
  parts[5]='tifu'     trong có 0 dấu * -> total=5
  parts[6]='l'        ngoài có 0 dấu * -> total=5
kết quả: 5


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def countAsterisks(self, s: str) -> int:
        bars = count = 0
        for ch in s:
            if ch == "|":
                bars += 1
            elif ch == "*" and bars % 2 == 0:
                count += 1
        return count

print(Solution().countAsterisks("l|*e*et|c**o|*de|"))
print(Solution().countAsterisks("iamprogrammer"))
print(Solution().countAsterisks("yo|uar|e**|b|e***au|tifu|l"))

2
0
5


## 10. Vì sao thuật toán đúng

- **Cặp theo thứ tự.** Theo đề, `|` thứ `2k+1` mở và `|` thứ `2k+2` đóng. Một vị trí nằm **trong** một cặp đúng khi số `|` phía trước nó là **lẻ** (đã mở mà chưa đóng); **ngoài** khi chẵn.
- **Đếm đúng.** Mỗi `*` được xét đúng một lần, tại thời điểm `bars` bằng số `|` đứng trước nó; ta cộng khi `bars` chẵn nên cộng đúng các `*` ngoài cặp.
- **`split`.** `split('|')` cắt thành `2k + 1` đoạn; đoạn thứ `j` nằm sau đúng `j` dấu `|`, vậy đoạn chỉ số chẵn là ngoài cặp — tương đương parity.
- **Số `|` chẵn** (đề đảm bảo) bảo đảm không có cặp bỏ dở; nếu lẻ thì `|` cuối cùng mở một cặp không có đóng và mọi thứ sau nó coi như trong cặp (parity vẫn cho kết quả hợp lý).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`<= 1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Cờ bool | `O(n)` | `O(1)` |
| 2. `split` + `[::2]` | `O(n)` | `O(n)` |
| 3. Parity `bars % 2` | `O(n)` | `O(1)` |
| 4. Regex xóa cặp | `O(n)` | `O(n)` |
| 5. `find` nhảy cặp | `O(n)` | `O(1)` |

Cả năm cách đều tuyến tính; chọn cách dễ đọc nhất (cờ hoặc parity). Không cách nào tốt hơn `O(n)` vì phải nhìn mỗi ký tự ít nhất một lần.

## 12. Bộ test

Đối chiếu tất cả các cách trên ví dụ đề, edge case và test ngẫu nhiên có `seed`.

In [16]:
import random

solutions = [SolutionFlag(), SolutionSplit(), SolutionParity(), SolutionRegex(), SolutionFind(), Solution()]

fixed = [
    ("l|*e*et|c**o|*de|", 2),
    ("iamprogrammer", 0),
    ("yo|uar|e**|b|e***au|tifu|l", 5),
    ("*", 1),                     # một dấu * và không có cặp
    ("**", 2),
    ("||", 0),                    # cặp rỗng
    ("|**|", 0),                  # tất cả * nằm trong cặp
    ("*|**|*", 2),                # * ở hai đầu nằm ngoài cặp
    ("|*|*|*|", 1),               # * thứ hai nằm giữa hai cặp
    ("a|b|c*d|e|*", 2),
]
for s, expected in fixed:
    for sol in solutions:
        assert sol.countAsterisks(s) == expected, (type(sol).__name__, s, expected)
print("đã qua", len(fixed), "test cố định cho", len(solutions), "cách")

đã qua 10 test cố định cho 6 cách


In [17]:
random.seed(2315)
for _ in range(1000):
    n = random.randint(1, 40)
    s = list(random.choice("ab**|") for _ in range(n))
    if s.count("|") % 2 == 1:                      # bảo đảm số | chẵn
        s[random.randrange(n)] = "a" if s.count("|") == 1 else "|"
    s = "".join(s)
    if s.count("|") % 2 == 1:
        s += "|"
    ref = SolutionFlag().countAsterisks(s)
    for sol in solutions[1:]:
        assert sol.countAsterisks(s) == ref, (type(sol).__name__, s)
print("1000 test ngẫu nhiên đối chiếu: OK")

1000 test ngẫu nhiên đối chiếu: OK


In [18]:
# Đối chiếu với định nghĩa trực tiếp: ghép cặp theo vị trí các dấu |
def truth(s):
    bar_pos = [i for i, ch in enumerate(s) if ch == "|"]
    inside = set()
    for k in range(0, len(bar_pos) - 1, 2):
        inside.update(range(bar_pos[k], bar_pos[k + 1] + 1))
    return sum(1 for i, ch in enumerate(s) if ch == "*" and i not in inside)

random.seed(15)
for _ in range(500):
    n = random.randint(1, 30)
    s = "".join(random.choice("a*|") for _ in range(n))
    if s.count("|") % 2:
        s += "|"
    assert Solution().countAsterisks(s) == truth(s), s
print("khớp định nghĩa ghép cặp theo vị trí: OK")

khớp định nghĩa ghép cặp theo vị trí: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm **tất cả** `*` (quên điều kiện ngoài cặp)

`s.count('*')` đếm cả `*` trong cặp. Ví dụ 1 sẽ ra `5` thay vì `2`.

In [19]:
s = "l|*e*et|c**o|*de|"
print("sai :", s.count("*"))
print("đúng:", Solution().countAsterisks(s))

sai : 5
đúng: 2


### Lỗi 2: Cập nhật cờ muộn hoặc quên `elif`

Nếu viết `if ch == '*' and not inside: count += 1` **trước** khi xử lý `|`, hoặc đổi cờ sai chỗ thì kết quả vẫn thường đúng — nhưng nếu đảo `inside` cho **mọi ký tự** thay vì chỉ `|` thì sai. Ví dụ dưới đảo cờ mỗi ký tự.

In [20]:
def sai_dao_moi_ky_tu(s):
    inside, count = False, 0
    for ch in s:
        inside = not inside                   # SAI: đảo ở mọi ký tự
        if ch == "*" and not inside:
            count += 1
    return count

print("sai :", sai_dao_moi_ky_tu("l|*e*et|c**o|*de|"), "| đúng:", Solution().countAsterisks("l|*e*et|c**o|*de|"))

sai : 2 | đúng: 2


### Lỗi 3: Hiểu sai "cặp" khi có nhiều gạch: `split` lấy đoạn lẻ thay vì chẵn

`s.split('|')[1::2]` là các đoạn **trong** cặp. Đây là lỗi lệch chỉ số hay gặp.

In [21]:
s = "l|*e*et|c**o|*de|"
sai = sum(p.count("*") for p in s.split("|")[1::2])
dung = sum(p.count("*") for p in s.split("|")[0::2])
print("sai (đoạn lẻ):", sai, "| đúng (đoạn chẵn):", dung)

sai (đoạn lẻ): 3 | đúng (đoạn chẵn): 2


### Lỗi 4: Regex tham lam `\|.*\|`

Mẫu `.*` khớp **cả** các `|` ở giữa nên xóa một mạch từ `|` đầu tới `|` cuối, kể cả phần ngoài cặp. Phải dùng `[^|]*` để dừng ở `|` kế tiếp.

In [22]:
import re

s = "l|*e*et|c**o|*de|"
sai = re.sub(r"\|.*\|", "", s).count("*")
dung = re.sub(r"\|[^|]*\|", "", s).count("*")
print("sai :", sai, "| đúng:", dung)

sai : 0 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "cặp mở/đóng ghép theo thứ tự" -> **cờ bật/tắt** hoặc **parity** của số dấu đã gặp.
- Đề muốn bỏ phần "bên trong" -> chuỗi bị `split` bởi dấu phân cách, lấy các đoạn chỉ số chẵn.
- Luôn thử ví dụ rỗng/biên: `"||"`, `"*"`, `"|**|"`.
- Regex mạnh nhưng dễ sai (tham lam vs không tham lam); dùng khi chắc chắn.
- Cùng pattern: Valid Parentheses, Remove Outermost Parentheses, bài đếm ký tự ngoài dấu ngoặc.

## 15. Biến thể của bài

- **Đếm `*` trong** cặp thay vì ngoài (đảo điều kiện).
- **Hai loại dấu** (`(`..`)` và `|`) lồng nhau: cần `depth` thay vì cờ.
- **Cặp lồng nhau** (như ngoặc): dùng biến độ sâu `depth += 1` khi mở, `-= 1` khi đóng; đếm khi `depth == 0`.
- **Trả về chuỗi sau khi xóa mọi cặp** (không đếm), như Cách 4.
- **Số dấu `|` lẻ:** quy ước cặp cuối bỏ dở; kiểm tra lại cách xử lý.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cờ bool | đảo `inside` mỗi `\|` | `O(n)` | `O(1)` | dễ hiểu nhất, người mới |
| 2. `split` | lấy đoạn chỉ số chẵn | `O(n)` | `O(n)` | code một dòng |
| 3. Parity | `bars % 2` | `O(n)` | `O(1)` | nộp bài, gọn |
| 4. Regex | xóa `\|[^\|]*\|` | `O(n)` | `O(n)` | xử lý chuỗi nhanh, đã quen regex |
| 5. `find` | nhảy qua từng cặp | `O(n)` | `O(1)` | ít duyệt ký tự trong cặp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().countAsterisks("*|*|*|*|*")` (có 4 dấu `|`).

**Bài 2 (tính tay).** Với `s = "a*b|c*|d**|e|*f"`, ghi các đoạn của `split('|')`, đánh dấu ngoài/trong, rồi đếm.

**Bài 3 (điền TODO).** Hoàn thành `count_inside(s)` đếm số `*` **trong** các cặp (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `remove_pairs(s)` trả về chuỗi sau khi xóa mọi cặp `|...|` (kể cả dấu `|`).

**Bài 5 (mini challenge).** Viết `count_with_depth(s)` cho chuỗi dùng `(` `)` có thể lồng nhau: đếm `*` ở độ sâu `0` (không nằm trong bất kỳ cặp ngoặc nào).

In [23]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def count_inside_todo(s):
    bars = count = 0
    # for ch in s: ...
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `*|*|*|*|*` có 5 đoạn; đoạn chẵn `0, 2, 4` mỗi đoạn một `*` -> 3. Bài 2: các đoạn chỉ số chẵn là `a*b`, `d**`, `*f`.
- **Mức 2 (vừa):** Bài 3: đếm `*` khi `bars % 2 == 1`. Bài 4: `re.sub(r"\|[^|]*\|", "", s)`.
- **Mức 3 (sâu):** Bài 5: `depth += 1` khi `(`, `depth -= 1` khi `)`, đếm `*` khi `depth == 0`. Chú ý khi cập nhật `depth` trước/sau.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
import re

# Bài 1 và 2
assert Solution().countAsterisks("*|*|*|*|*") == 3
assert Solution().countAsterisks("a*b|c*|d**|e|*f") == 1 + 2 + 1
parts = "a*b|c*|d**|e|*f".split("|")
assert parts == ["a*b", "c*", "d**", "e", "*f"]
assert sum(p.count("*") for p in parts[::2]) == 1 + 2 + 1

# Bài 3
def count_inside(s):
    bars = count = 0
    for ch in s:
        if ch == "|":
            bars += 1
        elif ch == "*" and bars % 2 == 1:
            count += 1
    return count

# Bài 4
def remove_pairs(s):
    return re.sub(r"\|[^|]*\|", "", s)

# Bài 5
def count_with_depth(s):
    depth = count = 0
    for ch in s:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        elif ch == "*" and depth == 0:
            count += 1
    return count

assert count_inside("l|*e*et|c**o|*de|") == 3
assert count_inside("abc") == 0
assert remove_pairs("a|bc|d|e|f") == "adf"
assert remove_pairs("abc") == "abc"
assert count_with_depth("*(*(*)*)*") == 2
assert count_with_depth("(**)") == 0
assert count_with_depth("**") == 2
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (parity của số dấu `|` đã gặp)

```python
class Solution:
    def countAsterisks(self, s: str) -> int:
        bars = count = 0
        for ch in s:
            if ch == "|":
                bars += 1
            elif ch == "*" and bars % 2 == 0:
                count += 1
        return count
```

**Checklist**

- [ ] Chỉ đếm `*` khi số `|` đã gặp là **chẵn** (ngoài cặp)?
- [ ] Dùng `elif` để `|` và `*` không bị xử lý trùng trong một ký tự?
- [ ] Cặp ghép theo thứ tự `1-2`, `3-4`, ... (không phải ghép lồng nhau)?
- [ ] Nếu dùng `split`, lấy đoạn chỉ số **chẵn** (`[::2]`)?
- [ ] Nếu dùng regex, mẫu là `\|[^|]*\|` (không tham lam)?
- [ ] Đã thử `"||"`, `"|**|"`, `"*"` và chuỗi không có `|`?
- [ ] Signature đúng `countAsterisks(self, s: str) -> int`?